# Sesión 07 — Lab Reto: Pipeline CDF Bronze → Silver
## Change Data Feed: Propagación Incremental en Dos Lotes

**Curso:** Databricks Data Engineer Associate  
**Runtime mínimo:** DBR 13.3 LTS  
**Plataforma:** Azure Databricks con Unity Catalog habilitado  
**Dataset:** `movimientos_suscripcion.csv`

---

### Contexto del reto

La tabla `dbassociate.bronze.movimientos` registra los movimientos de suscripciones de una plataforma digital: Altas, Upgrades, Downgrades, Bajas y Reactivaciones. La tabla Silver (`dbassociate.silver.movimientos`) debe mantenerse sincronizada con Bronze de forma incremental usando Change Data Feed.

Los datos llegan en **dos lotes**:
- **Lote 1:** Altas y Upgrades (MOV-001 al MOV-047)
- **Lote 2:** Downgrades, Bajas y Reactivaciones (MOV-048 al MOV-080)

### Lo que debes implementar

1. Crear `dbassociate.bronze.movimientos` con CDF habilitado
2. Insertar el Lote 1 y propagar a Silver
3. Insertar el Lote 2 **desde la versión correcta** (sin reprocesar el Lote 1)
4. Aplicar el Lote 2 a Silver manejando correctamente las Bajas (DELETE en Silver)

### Criterio de éxito
- Silver contiene el estado final correcto: Altas activas, Upgrades aplicados, Bajas eliminadas
- La celda de validación al final muestra conteos consistentes entre Bronze y Silver
- No hay registros duplicados en Silver

---

> **Pistas disponibles en cada TODO. Lee el enunciado completo antes de comenzar.**

In [0]:
# Configuración — celda completa, no modificar
CATALOGO      = "dbassociate"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
VOL_LANDING   = "/Volumes/dbassociate/default/vol_landing"
CSV_MOV       = f"{VOL_LANDING}/movimientos_suscripcion.csv"

TABLA_BRONZE  = f"{CATALOGO}.{SCHEMA_BRONZE}.movimientos"
TABLA_SILVER  = f"{CATALOGO}.{SCHEMA_SILVER}.movimientos"

try:
    dbutils.fs.ls(CSV_MOV)
    print(f"OK: {CSV_MOV}")
except Exception:
    raise FileNotFoundError(
        f"Subir movimientos_suscripcion.csv a {VOL_LANDING} antes de continuar."
    )

In [0]:
# Carga del CSV — celda completa, no modificar
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, DateType
from pyspark.sql import functions as F

schema_mov = StructType([
    StructField("movimiento_id",   StringType(),       nullable=False),
    StructField("suscripcion_id",  StringType(),       nullable=True),
    StructField("tipo_movimiento", StringType(),       nullable=True),
    StructField("plan_anterior",   StringType(),       nullable=True),
    StructField("plan_nuevo",      StringType(),       nullable=True),
    StructField("monto_anterior",  DecimalType(8, 2),  nullable=True),
    StructField("monto_nuevo",     DecimalType(8, 2),  nullable=True),
    StructField("fecha",           DateType(),         nullable=True),
    StructField("motivo",          StringType(),       nullable=True),
])

df_mov = (
    spark.read
    .option("header", "true")
    .schema(schema_mov)
    .csv(CSV_MOV)
)

print(f"Registros totales en el CSV: {df_mov.count()}")
print("\nDistribución por tipo de movimiento:")
display(
    df_mov.groupBy("tipo_movimiento").count().orderBy("count", ascending=False)
)

---
## TODO 1 — Crear la tabla Bronze con CDF habilitado y cargar el Lote 1

**Qué hacer:**
- Crear `dbassociate.bronze.movimientos` con el esquema dado (`schema_mov`)
- Habilitar Change Data Feed con TBLPROPERTIES
- Insertar solo los movimientos del Lote 1: donde `tipo_movimiento IN ('Alta', 'Upgrade')` y `movimiento_id <= 'MOV-047'`

**Pista:** Usar `CREATE TABLE ... TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')` en DDL, luego `df.write.format("delta").mode("append").saveAsTable(...)`

In [0]:
# TODO 1 — Implementar aquí

# Lote 1: Altas y Upgrades
df_lote1 = df_mov.filter(
    F.col("tipo_movimiento").isin(["Alta", "Upgrade"]) &
    (F.col("movimiento_id") <= "MOV-047")
)
print(f"Registros en Lote 1: {df_lote1.count()}")

# Crear tabla Bronze con CDF y cargar Lote 1
# ...

---
## TODO 2 — Crear la tabla Silver con el estado inicial del Lote 1

**Qué hacer:**
- Leer el CDF de Bronze desde la versión 1 (la versión 0 es la carga inicial que irá directo a Silver)
- Crear `dbassociate.silver.movimientos` con los datos del Lote 1 (versión 0 de Bronze)
- Aplicar el CDF del Lote 1 con MERGE INTO Silver

**Pista:**
- Leer CDF: `spark.read.format("delta").option("readChangeFeed", "true").option("startingVersion", 1).table(TABLA_BRONZE)`
- Columnas del CDF: `_change_type`, `_commit_version`, `_commit_timestamp` + columnas originales
- Para la propagación: filtrar `_change_type IN ('insert', 'update_postimage')`

In [0]:
# TODO 2 — Implementar aquí

# Crear Silver con los datos del Lote 1
# Leer CDF desde versión 1
# Aplicar upserts a Silver
# ...

---
## TODO 3 — Guardar el watermark (última versión procesada del Lote 1)

**Qué hacer:**
- Obtener el `max(_commit_version)` del CDF del Lote 1
- Guardar ese valor en una variable `watermark_lote1`

**Por qué importa:** Al procesar el Lote 2, llamarás a `table_changes(TABLA_BRONZE, watermark_lote1 + 1)` para leer **solo** los cambios nuevos, sin reprocesar el Lote 1.

**Pista:** `df_cdf_lote1.agg(F.max("_commit_version")).collect()[0][0]`

In [0]:
# TODO 3 — Implementar aquí

# watermark_lote1 = ...
# print(f"Última versión del Lote 1: {watermark_lote1}")

---
## TODO 4 — Insertar el Lote 2 en Bronze

**Qué hacer:**
- El Lote 2 contiene: Downgrades (`MOV-048` a `MOV-060`), Bajas (`MOV-061` a `MOV-065`), Reactivaciones y el resto hasta `MOV-080`
- Insertar el Lote 2 en `dbassociate.bronze.movimientos` con mode `append`

**Pista:** `df_lote2 = df_mov.filter(F.col("movimiento_id") > "MOV-047")`

In [0]:
# TODO 4 — Implementar aquí

# df_lote2 = ...
# Insertar en Bronze
# ...

---
## TODO 5 — Leer el CDF del Lote 2 desde la versión correcta

**Qué hacer:**
- Leer el CDF desde `watermark_lote1 + 1` para obtener **solo** los cambios del Lote 2
- Mostrar la distribución de `_change_type` para el Lote 2

**Punto clave:** Si lees desde la versión 1 en lugar de `watermark_lote1 + 1`, también reprocesarás el Lote 1.

**Pista:** `spark.read.format("delta").option("readChangeFeed", "true").option("startingVersion", watermark_lote1 + 1).table(TABLA_BRONZE)`

In [0]:
# TODO 5 — Implementar aquí

# df_cdf_lote2 = ...
# Mostrar distribución por _change_type
# ...

---
## TODO 6 — Aplicar el Lote 2 a Silver

**Qué hacer:**
- Aplicar upserts (`insert` + `update_postimage`) a Silver con MERGE INTO
- Aplicar deletes (`delete`) a Silver con MERGE INTO con WHEN MATCHED DELETE

**Punto clave:** El Lote 2 contiene Bajas (`tipo_movimiento = 'Baja'`). En Bronze esto se registra como un INSERT (nuevo registro de movimiento). En Silver, las Bajas **también** se insertan como registro de movimiento — la tabla Silver en este contexto es un log de movimientos, no el estado actual de la suscripción.

**Pista:** El campo de join para el MERGE es `movimiento_id` (identificador único del movimiento).

In [0]:
# TODO 6 — Implementar aquí

# Aplicar upserts del Lote 2 a Silver
# Aplicar deletes del Lote 2 a Silver (si los hay)
# ...

---
## Validación — Celda completa, no modificar

Si la implementación es correcta:
- Silver debe tener el mismo número de registros que Bronze (todos los movimientos)
- La distribución por `tipo_movimiento` debe ser idéntica en Bronze y Silver
- No debe haber duplicados (cada `movimiento_id` aparece exactamente una vez)

In [0]:
# Validación — no modificar
bronze_count = spark.table(TABLA_BRONZE).count()
silver_count = spark.table(TABLA_SILVER).count()

print(f"Registros en Bronze: {bronze_count}")
print(f"Registros en Silver: {silver_count}")
print(f"Diferencia: {bronze_count - silver_count}")

if bronze_count == silver_count:
    print("CORRECTO: Bronze y Silver tienen el mismo número de registros.")
else:
    print("REVISAR: El número de registros no coincide.")

print("\n=== Distribución por tipo de movimiento ===")
print("Bronze:")
display(spark.table(TABLA_BRONZE).groupBy("tipo_movimiento").count().orderBy("tipo_movimiento"))

print("Silver:")
display(spark.table(TABLA_SILVER).groupBy("tipo_movimiento").count().orderBy("tipo_movimiento"))

print("\n=== Verificación de duplicados en Silver ===")
duplicados = spark.table(TABLA_SILVER).groupBy("movimiento_id").count().filter("count > 1")
dup_count = duplicados.count()
if dup_count == 0:
    print("CORRECTO: No hay duplicados en Silver.")
else:
    print(f"REVISAR: {dup_count} movimiento_id con duplicados en Silver:")
    display(duplicados)

In [0]:
# LIMPIEZA — descomentar para restablecer el ambiente
# spark.sql(f"DROP TABLE IF EXISTS {TABLA_BRONZE}")
# spark.sql(f"DROP TABLE IF EXISTS {TABLA_SILVER}")
# print("Limpieza completada")